# Arm 1: baseline under the fixed-epoch protocol, remaining folds (2, 3, 4) x seeds 42/43/44

Completes the 5-fold x 3-seed baseline (folds 0 and 1 already done in notebook 02). 100 epochs, no early stopping, last.pt is the primary metric.
Settings: Accelerator *GPU T4 x2*, Internet ON. The code path is the one already smoke-tested, so `SMOKE` defaults to False: use *Save Version -> Save & Run All*.


In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/nithoukhan1/hbfmid.git"
REPO = "/kaggle/working/hbfmid"
if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.215"], check=True)
os.environ["WANDB_MODE"] = "disabled"
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__)
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU T4 x2, then restart."
print("repo commit:", subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())


In [ ]:
DATA_ROOT = "/kaggle/input/datasets/nithoukhan2/hbfmid/Human Bone Fractures Multi-modal Image Dataset (HBFMID)/Bone Fractures Detection"
assert os.path.isdir(DATA_ROOT), DATA_ROOT
SMOKE = False
MODEL = "yolov8s.pt"
USE_ALL_GPUS = True
if SMOKE:
    EPOCHS, OUT = 2, "/kaggle/working/results/baseline_fixed_smoke"
    PLAN = [("clean", 2, 42)]
else:
    EPOCHS, OUT = 100, "/kaggle/working/results/baseline_fixed"
    PLAN = [("clean", f, s) for f in (2, 3, 4) for s in (42, 43, 44)]     # arm 1, folds 2-4, 3 seeds
print(len(PLAN), "runs, epochs =", EPOCHS, "->", OUT)


In [ ]:
import queue, threading, time
os.makedirs(f"{OUT}/logs", exist_ok=True)
gpus = list(range(torch.cuda.device_count())) if USE_ALL_GPUS else [0]
q = queue.Queue()
for t in PLAN:
    q.put(t)
failed = []

def worker(gpu):
    while True:
        try:
            mode, fold, seed = q.get_nowait()
        except queue.Empty:
            return
        name = f"{mode}_f{fold}_s{seed}"
        cmd = [sys.executable, f"{REPO}/scripts/training/run_baseline.py",
               "--mode", mode, "--fold", str(fold), "--seed", str(seed), "--model", MODEL,
               "--epochs", str(EPOCHS), "--fixed-epochs", "--device", str(gpu), "--data-root", DATA_ROOT,
               "--splits-dir", f"{REPO}/data_splits", "--out", OUT]
        t0 = time.time()
        with open(f"{OUT}/logs/{name}.log", "w") as f:
            rc = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT).returncode
        print(f"[GPU {gpu}] {name}: {'OK' if rc == 0 else 'FAILED rc=' + str(rc)} in {(time.time()-t0)/60:.1f} min", flush=True)
        if rc != 0:
            failed.append(name)

threads = [threading.Thread(target=worker, args=(g,)) for g in gpus]
[t.start() for t in threads]; [t.join() for t in threads]
print("failed runs:", failed or "none")
for n in failed:
    print("\n==== tail of", n)
    print("".join(open(f"{OUT}/logs/{n}.log").readlines()[-25:]))


In [ ]:
import shutil
subprocess.run([sys.executable, f"{REPO}/scripts/evaluation/aggregate_arm.py", "--name", "arm1_baseline", "--runs", OUT, f"{REPO}/results/controls", "--out", OUT])
shutil.make_archive("/kaggle/working/baseline_fixed_results", "zip", OUT)
print(sorted(os.listdir(OUT)))
if failed:
    print("\nWARNING: failed runs:", failed)


After the run: download `baseline_fixed_results.zip`, unzip into `results/baseline_fixed/`, commit and push.
